# NCTBench-Pilot — Enhanced Results (All 18 Configs)

Three enhancements applied on top of the original 12 configs:
- **A** — Cross-encoder reranking + MMR (configs 13–18)
- **B** — Stricter grounded prompt (eliminates hallucination)
- **C** — ROUGE-L + BERTScore + std/CI95 + subject breakdown

Run order:
1. `python evaluation/rescore_existing.py`   ← re-score configs 1-12 with new metrics (no API)
2. `python evaluation/run_evaluation_enhanced.py`  ← run configs 13-18 (~2.5 hrs, ~$2)
3. This notebook visualizes all results

In [ ]:
import sys, json, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

sys.path.insert(0, str(Path(r'E:\CSAA Project\pipeline')))
sys.path.insert(0, str(Path(r'E:\CSAA Project\evaluation')))

RESULTS_DIR = Path(r'E:\CSAA Project\evaluation\results')
NOTEBOOKS   = Path(r'E:\CSAA Project\notebooks')

# Load enriched summary (all 18 configs)
all_csv = RESULTS_DIR / 'enriched_summary_all.csv'
base_csv = RESULTS_DIR / 'enriched_summary.csv'

if all_csv.exists():
    df = pd.read_csv(all_csv)
    print(f'Loaded enriched_summary_all.csv — {len(df)} configs')
elif base_csv.exists():
    df = pd.read_csv(base_csv)
    print(f'Loaded enriched_summary.csv (configs 1-12 only) — {len(df)} configs')
    print('Run run_evaluation_enhanced.py to add configs 13-18')
else:
    df = pd.read_csv(RESULTS_DIR / 'summary_table.csv')
    print('Loaded original summary_table.csv (no ROUGE-L/BERTScore yet)')

print(df[['config_id','retriever','generator','prompt_type','token_f1']].to_string(index=False))

## 1. Three-Metric Comparison: Token F1 vs ROUGE-L vs BERTScore

In [ ]:
grounded = df[df['prompt_type'] == 'grounded'].copy() if 'prompt_type' in df.columns else df.copy()

has_rouge  = 'rouge_l'    in grounded.columns
has_bert   = 'bert_score' in grounded.columns

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

cfg_labels = [f"C{int(r['config_id'])}\n{r.get('retriever','')[:6]}" 
              for _, r in grounded.iterrows()]
x = range(len(grounded))
w = 0.25

# Panel 1: three metrics side by side
ax = axes[0]
ax.bar([i - w for i in x], grounded['token_f1'],    width=w, label='Token F1',   color='#0072B2')
if has_rouge:
    ax.bar(x,                   grounded['rouge_l'],     width=w, label='ROUGE-L',    color='#E69F00')
if has_bert:
    ax.bar([i + w for i in x],  grounded['bert_score'],  width=w, label='BERTScore',  color='#009E73')
ax.set_xticks(list(x))
ax.set_xticklabels(cfg_labels, fontsize=7)
ax.set_title('Grounded Configs: All Three Metrics')
ax.set_ylabel('Score')
ax.set_ylim(0, 0.7)
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel 2: ROUGE-L gap (ROUGE-L - Token F1 shows paraphrase recovery)
ax2 = axes[1]
if has_rouge:
    gap = (grounded['rouge_l'] - grounded['token_f1']).values
    colors = ['#009E73' if g > 0 else '#CC0000' for g in gap]
    ax2.bar(list(x), gap, color=colors)
    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(cfg_labels, fontsize=7)
    ax2.set_title('ROUGE-L − Token F1\n(+ve = model paraphrases correctly)')
    ax2.set_ylabel('Gap')
else:
    ax2.text(0.5, 0.5, 'Run rescore_existing.py\nto get ROUGE-L', 
             ha='center', va='center', transform=ax2.transAxes)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Panel 3: hallucination rate by config
ax3 = axes[2]
if 'hallucination_rate' in grounded.columns:
    hall = grounded['hallucination_rate'].values
    cmap = ['#CC0000' if h > 0.4 else ('#E69F00' if h > 0.2 else '#009E73') for h in hall]
    ax3.bar(list(x), hall, color=cmap)
    ax3.axhline(0.2, color='gray', linestyle='--', linewidth=0.8, label='20% threshold')
    ax3.set_xticks(list(x))
    ax3.set_xticklabels(cfg_labels, fontsize=7)
    ax3.set_title('Hallucination Rate\n(lower is better)')
    ax3.set_ylabel('Rate')
    ax3.set_ylim(0, 0.7)
    ax3.legend(fontsize=8)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

plt.suptitle('NCTBench-Pilot — Enhanced Results (All Grounded Configs)', fontsize=13)
plt.tight_layout()
plt.savefig(NOTEBOOKS / 'enhanced_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → notebooks/enhanced_results.png')

## 2. Subject Breakdown: ICT vs Science

In [ ]:
if 'ict_f1' in df.columns and 'science_f1' in df.columns:
    grounded = df[df['prompt_type'] == 'grounded'].copy() if 'prompt_type' in df.columns else df.copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Token F1 by subject
    x = range(len(grounded))
    cfg_labels = [f"C{int(r['config_id'])}" for _, r in grounded.iterrows()]
    w = 0.35
    axes[0].bar([i - w/2 for i in x], grounded['ict_f1'],     width=w, label='ICT',     color='#0072B2')
    axes[0].bar([i + w/2 for i in x], grounded['science_f1'], width=w, label='Science',  color='#E69F00')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(cfg_labels)
    axes[0].set_title('Token F1: ICT vs Science')
    axes[0].set_ylabel('Token F1')
    axes[0].legend()
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    # ROUGE-L by subject
    if 'ict_rouge_l' in grounded.columns:
        axes[1].bar([i - w/2 for i in x], grounded['ict_rouge_l'],     width=w, label='ICT',    color='#0072B2')
        axes[1].bar([i + w/2 for i in x], grounded['science_rouge_l'], width=w, label='Science', color='#E69F00')
        axes[1].set_xticks(list(x))
        axes[1].set_xticklabels(cfg_labels)
        axes[1].set_title('ROUGE-L: ICT vs Science')
        axes[1].set_ylabel('ROUGE-L')
        axes[1].legend()
        axes[1].spines['top'].set_visible(False)
        axes[1].spines['right'].set_visible(False)

    plt.suptitle('Subject-Level Performance Breakdown', fontsize=13)
    plt.tight_layout()
    plt.savefig(NOTEBOOKS / 'subject_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Subject breakdown not available. Run rescore_existing.py first.')

## 3. Statistical Summary with Confidence Intervals

In [ ]:
if 'token_f1_ci95' in df.columns:
    grounded = df[df['prompt_type'] == 'grounded'].copy() if 'prompt_type' in df.columns else df.copy()
    grounded = grounded.sort_values('token_f1', ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(grounded))
    f1s  = grounded['token_f1'].values
    ci95 = grounded['token_f1_ci95'].values
    labels = [f"C{int(r['config_id'])} {r.get('desc', r.get('retriever',''))[:18]}" 
               for _, r in grounded.iterrows()]

    ax.barh(list(x), f1s, xerr=ci95, color='#0072B2', alpha=0.8,
            error_kw={'ecolor': 'black', 'capsize': 4, 'linewidth': 1.2})
    ax.set_yticks(list(x))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Token F1 (± 95% CI)')
    ax.set_title('Token F1 with 95% Confidence Intervals\n(sorted by performance)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(NOTEBOOKS / 'ci95_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('CI95 not available yet. Run rescore_existing.py first.')

## 4. Full Results Table (All 18 Configs)

In [ ]:
display_cols = [
    'config_id', 'retriever', 'generator', 'prompt_type',
    'recall_at_3', 'token_f1', 'rouge_l', 'bert_score',
    'hallucination_rate', 'bn_f1', 'en_f1',
    'ict_f1', 'science_f1', 'token_f1_ci95'
]
show = [c for c in display_cols if c in df.columns]
print(df[show].to_string(index=False))

print('\n--- BEST CONFIGS BY METRIC ---')
for metric in ['token_f1', 'rouge_l', 'bert_score']:
    if metric in df.columns:
        best = df.loc[df[metric].idxmax()]
        print(f"  Best {metric:15s}: Config {int(best['config_id']):2d} "
              f"({best.get('desc', best.get('retriever',''))}) = {best[metric]:.4f}")

## 5. Run Rescore (Enhancement C) — No API Calls

In [ ]:
import subprocess
result = subprocess.run(
    [r'C:\Users\shiha\AppData\Local\anaconda3\envs\torch_env\python.exe',
     str(Path(r'E:\CSAA Project\evaluation\rescore_existing.py'))],
    capture_output=True, text=True, cwd=r'E:\CSAA Project\evaluation'
)
out = result.stdout
print(out[-4000:] if len(out) > 4000 else out)
if result.returncode != 0:
    print('ERRORS:', result.stderr[-2000:])

## 6. Run Enhanced Evaluation (Configs 13-18) — Uses API (~$2, ~2.5 hrs)

In [ ]:
import subprocess
result = subprocess.run(
    [r'C:\Users\shiha\AppData\Local\anaconda3\envs\torch_env\python.exe',
     str(Path(r'E:\CSAA Project\evaluation\run_evaluation_enhanced.py'))],
    capture_output=True, text=True, cwd=r'E:\CSAA Project\evaluation'
)
out = result.stdout
print(out[-6000:] if len(out) > 6000 else out)
if result.returncode != 0:
    print('ERRORS:', result.stderr[-2000:])